In [30]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY2")

if not api_key:
    raise ValueError("API Key not found in .env")

client = genai.Client(api_key=api_key)

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0)

# **CHAIN WITH Parallel Chains**

In [31]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")])

In [32]:
# TASK - 2 [LLM]
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0)

In [33]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [34]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

### **Parallel Chain 1**

In [35]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]


llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0)

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

### **Parallel Chain 2**

In [36]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [37]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])
    
    # TASK - 2 [LLM]
    
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.8-flash",
        temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)


### **Final Orchestration**

In [38]:
final_chain = ( 
                    prompt_template | 
                    llm | 
                    str_parser | 
                    dictionary_maker_runnable |
                    RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)
    

In [39]:
final_chain.invoke("KGF")

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 17h57m53.395235699s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

## **Chain as a Runnable**

In [ ]:
# TASK - 1 [Beautify Function]

def beautify(final_response:dict)-> dict:

    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)


# TASK - 2 [Final Chain]

# final_chain 


# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")


{'linkedin': 'Just watched Pushpa: The Rise (2021), directed by Sukumar — a gritty Telugu action‑drama that’s as much about survival and ambition as it is about crime.\n\nQuick snapshot:\n- Follows Pushpa Raj (Allu Arjun), a lowly labourer in the Seshachalam forests who claws his way up the red sandalwood trade through grit, cunning and violence.\n- As he gains power he clashes with rivals, corrupt officials and the law, while negotiating a fraught personal relationship with Srivalli.\n- Noted for a standout lead performance, driving music and high‑energy action.\n\nTakeaways for professionals and storytellers:\n- Ambition + resourcefulness can change trajectories — but so do the choices and trade‑offs along the way.\n- Systems matter: understanding informal rules, incentives and power structures is crucial when operating in complex environments.\n- Leadership is messy: gaining influence brings conflicts, ethical dilemmas and the need to manage alliances.\n- Craft counts: execution — p